# Predicción de accidentalidad — El Poblado

Punto de partida para la parte de leaderboard. Fija únicamente cómo conectarse a los datos y en qué formato debe quedar la entrega.

La construcción del target, la limpieza, la partición de validación y la ingeniería de características las desarrollan ustedes (secciones 4.1 a 4.4 del taller).

Lean el README del reto antes de empezar.

## 1. Conexión a los datos

In [ ]:
# Descarga la base de datos si todavía no está en la carpeta (~85 MB).
# Funciona igual en Windows, macOS y Linux: nada de comandos de terminal.
import urllib.request
from pathlib import Path

DB_PATH = Path("data_accidentes_poblado.sqlite3")

if not DB_PATH.exists():
    URL = ("https://d3qixogk4zgixq.cloudfront.net/data/"
           "prediccion-accidentalidad-poblado/data_accidentes_poblado.sqlite3")
    print("Descargando…  (una sola vez, puede tardar)")
    urllib.request.urlretrieve(URL, DB_PATH)

print(f"{DB_PATH} — {DB_PATH.stat().st_size / 1e6:.0f} MB")

In [ ]:
import sqlite3

import pandas as pd

con = sqlite3.connect(DB_PATH)
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con))
# Exploren cada tabla: qué representa una fila, qué llaves las cruzan y
# qué rango de fechas cubre cada una. Revisen también su consistencia
# entre sí (sección 4.2 del taller).
con.close()

## 2. Su trabajo

En su propio notebook o a partir de aquí:

- EDA y calidad de datos
- Definición y construcción de la variable objetivo
- Unión de tablas e ingeniería de características
- Estrategia de validación (¿por qué una partición aleatoria sería un error aquí?)
- Manejo del desbalance, comparación de modelos, selección final

El README explica la regla que deben respetar al construir variables.

## 3. Formato de la entrega

Esto sí es fijo. El archivo debe tener exactamente las columnas `id` y `target`, y una fila por cada pareja (barrio, hora) del período de evaluación.

- `id`: `"{BARRIO}|{TW}"`, con `TW` como `YYYY-MM-DD HH:MM:SS`
- `target`: probabilidad continua entre 0 y 1

In [ ]:
def construir_submission(barrios, marcas_tiempo, probabilidades, ruta="mi_prediccion.csv"):
    """Arma el CSV de entrega en el formato que espera el leaderboard."""
    sub = pd.DataFrame({
        "id": pd.Series(barrios).astype(str)
        + "|"
        + pd.to_datetime(pd.Series(marcas_tiempo)).dt.strftime("%Y-%m-%d %H:%M:%S"),
        "target": probabilidades,
    })
    if len(sub) != 80256:
        raise ValueError(f"Se esperaban 80256 filas, hay {len(sub)}")
    if sub["id"].duplicated().any():
        raise ValueError("Hay ids duplicados")
    if not sub["target"].between(0, 1).all():
        raise ValueError("target debe estar entre 0 y 1")
    sub.to_csv(ruta, index=False)
    return sub


# construir_submission(test["BARRIO"], test["TW"], mis_probabilidades)